## docling+ pdfs testing

In [ ]:
from docling.document_converter import DocumentConverter

# For a local PDF file
source_path = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/autorizacoes/autorizacao-bradesco-115013010-1738981574303.pdf"

# Or for a PDF from a URL
# source_url = "https://example.com/document.pdf"

converter = DocumentConverter()
result = converter.convert(source_path, OC) # or source_url

# Export to Markdown (or other formats like JSON, plain text)
markdown_output = result.document.export_to_markdown()
print(markdown_output)

<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>autorizacao-bradesco-115013010-1738981574303</title>
<meta name="generator" content="Docling HTML Serializer">
<style>
    html {
        background-color: #f5f5f5;
        font-family: Arial, sans-serif;
        line-height: 1.6;
    }
    body {
        max-width: 800px;
        margin: 0 auto;
        padding: 2rem;
        background-color: white;
        box-shadow: 0 0 10px rgba(0,0,0,0.1);
    }
    h1, h2, h3, h4, h5, h6 {
        color: #333;
        margin-top: 1.5em;
        margin-bottom: 0.5em;
    }
    h1 {
        font-size: 2em;
        border-bottom: 1px solid #eee;
        padding-bottom: 0.3em;
    }
    table {
        border-collapse: collapse;
        margin: 1em 0;
        width: 100%;
    }
    th, td {
        border: 1px solid #ddd;
        padding: 8px;
        text-align: left;
    }
    th {
        background-color: #f2f2f2;
        font-weight: bold;
    }
    figure {
        margin: 1.5em 0;
 

In [14]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TesseractCliOcrOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

source_path = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/autorizacoes/autorizacao-bradesco-115013010-1738981574303.pdf"

# 1) Configure OCR (Portuguese). Don't force full-page OCR so native text is preserved.
ocr_opts = TesseractCliOcrOptions(lang=["por"])  # or ["auto"] to auto-detect

# 2) Build pipeline with OCR enabled `pipeline_options.ocr_options.tesseract_cmd='tesseract'
pipe = PdfPipelineOptions(do_ocr=True, ocr_options=ocr_opts)
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipe)}
)

# 3) Convert + export
result = converter.convert(source_path)
markdown_output = result.document.export_to_markdown()
print(markdown_output)


## Bradesco Seguros

2 - N° Guia no Prestador

## Guia de Solicitaçªo de Internaçªo

- 10 - Nome

Dados do BeneficiÆrio

5 - Senha

- 1 - Registro ANS

- 3 - Nœmero da Guia Atribuído pela Operadora

- 4 - Data da Autorizaçªo

Procedimentos ou Itens Assistenciais Adicionais Solicitados

- 7 - Nœmero da Carteira

- 28 - Indicaçªo Clínica

- 6 - Data de Validade da Senha

- 8 - Validade da Carteira

- 9 - Atendimento a RN

- 50 - Nome Social

Dados do Contratado Solicitante

- 12 - Código na Operadora 30910

- 13 - Nome do Contratado

- 15 - Conselho Profissional

- 14 - Nome do Profissional Solicitante

- 16 - Nœmero do Conselho

17 - UF

- 18 - Código  CBO

Dados do Hospital / Local Solicitado / Dados da Internaçªo

- 19 - Código na Operadora / CNPJ

- 20 - Nome do Hospital/ Local Solicitado

- 21 - Data Sugerida para Internaçªo (Real)

- 23 -Tipo de Internaçªo

- 24 - Regime de Internaçªo

- 25 - Qtde. DiÆrias Solicitadas

- 26 - Previsªo de uso de OPME

- 27 - Previsªo de uso de Quimi

# unstructured

In [21]:
from unstructured.partition.pdf import partition_pdf

source_path = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/autorizacoes/autorizacao-sulamerica-185374517-1736364730349.pdf"

# Extrai blocos de texto do PDF
elements = partition_pdf(filename=source_path, languages=["portuguese"])

# Junta o texto
full_text = "\n".join([el.text for el in elements if el.text])

print(full_text)

VPP - Validação Prévia de Procedimentos Solicitação de Internação
Dados da Solicitação Código na Operadora 100000018457 Nº da Guia 185374517 Senha 5040290770 Status Autorizado parcialmente
Referenciado HOSPITAL MATER DEI Nº da Guia Principal
Status da Senha Pendente de Confirmação
Descrição do Motivo
Nº da Guia no Prestador 0 Data da Autorização da VPP 17/12/2024
CNES 7684878 Data da Solicitação 16/12/2024 Data de Validade da VPP 07/02/2025
Dados do Beneficiário (Segurado) Nome MARCELO MORELLI CARRIERI Plano ESPECIAL 100 Carteira do Beneficiário 557 88888 4753 3651 0015
Data de Nascimento 20/05/1962 Produto 557 - PME AMB HOSP C OBST ADAPTADO
Sexo Masculino
Recém Nato Não
Dados do Atendimento Data Prevista da Internação 17/12/2024 Regime de Internação Hospitalar
Data Efetiva da Internação Data da Alta
Tipo de Acomodação APARTAMENTO STANDARD
Caráter do Atendimento Eletivo Diárias Solicitadas 01
Tipo de Internação Cirúrgica Diárias Autorizadas 01
Procedimentos
Nº
1
Código
Descrição
307150

## success on unstructured. looking at textract now

In [22]:
# trying to use the mudular code to get the workflow feeling
import sys

# add app to path so we dont get the error ModuleNotFoundError: No module named 'app'
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from app.utils.process_images import process_blobs
from app.utils.textract_service import TextractKVExtractor, extract_text_single_id
from app.utils.llm_service import AnthropicLLMService
from app.utils.aws_services_handler import create_boto3_client
from app.utils.config import load_config, AppConstants
from app.utils.logger import get_logger
from app.utils.system_prompts.prompt_handler import Prompts


logger = get_logger(name=__name__)
config_vars = load_config()

In [23]:
app_constants = AppConstants()
texttract_client = create_boto3_client("textract", config_vars)


texttract_instance = TextractKVExtractor(texttract_client)


{"timestamp": "2025-08-19T10:37:17", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente TEXTRACT para a região: us-east-1...", "filename": "aws_services_handler.py", "lineno": 17}
{"timestamp": "2025-08-19T10:37:17", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Cliente TEXTRACT criado com sucesso.", "filename": "aws_services_handler.py", "lineno": 26}


In [25]:
# PROCESSING CUSTOM D=DICT WITH IDS AND BLOBS
# blob = load the pdf blob to bytes
source_path = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/autorizacoes/autorizacao-sulamerica-185374517-1736364730349.pdf"
with open(source_path, "rb") as f:
    blob_1 = f.read()

ids_and_blobs = {
    "id1": blob_1,
}
byte_png_images_and_ids = process_blobs(ids_and_blobs, img_enhancement=False, MAX_IMAGES_PER_BLOB=20)
logger.info(f"Processed {len(byte_png_images_and_ids)} byte PNG images.")

{"timestamp": "2025-08-19T10:47:25", "level": "INFO", "name": "app.utils.process_images", "message": "Converting BLOB id1 to images...", "filename": "process_images.py", "lineno": 70}
{"timestamp": "2025-08-19T10:47:25", "level": "INFO", "name": "app.utils.process_images", "message": "Processando PDF com 2 página(s)...", "filename": "process_images.py", "lineno": 23}
{"timestamp": "2025-08-19T10:47:25", "level": "INFO", "name": "app.utils.process_images", "message": "✅ Converted image from BLOB id1 to png bytes.", "filename": "process_images.py", "lineno": 80}
{"timestamp": "2025-08-19T10:47:25", "level": "INFO", "name": "__main__", "message": "Processed 1 byte PNG images.", "filename": "1596124332.py", "lineno": 11}


In [26]:
textract_results = {}
for id in byte_png_images_and_ids:
    single_id_full_text = extract_text_single_id(images_bytes_list=byte_png_images_and_ids[id],
                                                 textract_instance=texttract_instance, extract_full_text=False)

    textract_results[id] = single_id_full_text

In [27]:
textract_results

{'id1': "defaultdict(<class 'list'>, {'Produto': ['557 PME AMB HOSP c OBST ADAPTADO'], 'Status': ['Autorizado parcialmente'], 'Plano': ['ESPECIAL 100'], 'Emitido em:': ['08/01/2025 16:32:10'], 'Nome': ['MARCELO MORELLI CARRIERI'], 'Tipo de Internação': ['Cirúrgica'], 'Itens Assistencias': [''], 'Nº da Guia no Prestador': ['0'], 'Mensagens ao Prestador': [''], 'Data Prevista da Internação': ['17/12/2024'], 'Data da Alta': [''], 'N° da Guia': ['185374517'], 'Descrição do Motivo': [''], 'Senha': ['5040290770'], 'Data da Solicitação': ['16/12/2024'], 'Status da Senha': ['Pendente de Confirmação'], 'Diárias Solicitadas': ['01'], 'Referenciado': ['HOSPITAL MATER DEI'], 'Recém Nato': ['Não'], 'Sexo': ['Masculino'], 'Código na Operadora': ['100000018457'], 'Data de Validade da VPP': ['07/02/2025'], 'Tipo de Acomodação': ['APARTAMENTO STANDARD'], 'Data da Autorização da VPP': ['17/12/2024'], 'Diárias Autorizadas': ['01'], 'Pagina': ['1 de 2'], 'Data de Nascimento': ['20/05/1962'], 'CNES': ['768